# 🤖 A2A (Agent-to-Agent) デモ
## Google Colab + HuggingFace Transformers (ローカルLLM)

### このノートブックでできること
- **Transformers** でローカルLLMをColab上で動かす（`Qwen2.5-0.5B-Instruct`）
- **4体のエージェント**（オーケストレーター + ワーカー×3）をFlaskサーバーとして起動
- **エージェントカード** を `/.well-known/agent.json` で公開し、HTTPで取得・print表示の両方を実施
- エージェント同士が **A2Aプロトコル** (tasks/send, tasks/get) でタスクを委譲し合うデモを実行

### アーキテクチャ
```
ユーザー
  └─→ OrchestratorAgent (port 8100)
          │  起動時にエージェントカードを収集
          ├─→ SummarizerAgent  (port 8101)  ← 要約担当
          ├─→ TranslatorAgent  (port 8102)  ← 翻訳担当
          └─→ KeywordAgent     (port 8103)  ← キーワード抽出担当
```

### 使用モデル
| 項目 | 内容 |
|---|---|
| モデル | `Qwen/Qwen2.5-0.5B-Instruct` |
| サイズ | ~1 GB |
| ランタイム | T4 GPU (Colab無料枠) |
| 推奨ランタイム | **GPU** (Runtime > Change runtime type > T4 GPU) |

> ⚠️ **事前にGPUランタイムを選択してください**: Runtime > Change runtime type > T4 GPU

## Step 1: ライブラリのインストール

In [ ]:
!pip install transformers accelerate flask flask-cors --quiet
print("✅ Dependencies installed")

## Step 2: ローカルLLMの初期化（Transformers）

モデルは全エージェントで**共有**します（メモリ節約のため）。  
実際のA2A環境ではエージェントごとに異なるモデルを持つことが多いですが、  
Colabの制約上1モデルをシステムプロンプトで役割分けします。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"📥 Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"✅ Model loaded on: {model.device}")

def llm_call(user_prompt: str, system_prompt: str = "", max_new_tokens: int = 256) -> str:
    """Transformers pipeline経由でローカル推論を実行する"""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding（再現性重視）
            pad_token_id=tokenizer.eos_token_id,
        )

    # 入力トークンを除いた生成部分だけをデコード
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# 動作確認
test = llm_call("Say hello in one sentence.", max_new_tokens=32)
print(f"\nLLM test: {test}")

## Step 3: A2Aプロトコルの型定義

Google A2A仕様に準拠したデータ構造を定義します。

In [ ]:
import json
import uuid
import threading
import time
import requests
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Dict
from datetime import datetime, timezone
from flask import Flask, request, jsonify
from flask_cors import CORS

# ── AgentCard ────────────────────────────────────────────────────────
@dataclass
class AgentCapabilities:
    streaming: bool = False
    pushNotifications: bool = False
    stateTransitionHistory: bool = True

@dataclass
class AgentSkill:
    id: str
    name: str
    description: str
    inputModes: List[str]  = field(default_factory=lambda: ["text"])
    outputModes: List[str] = field(default_factory=lambda: ["text"])

@dataclass
class AgentCard:
    name: str
    description: str
    url: str
    version: str
    capabilities: AgentCapabilities
    skills: List[AgentSkill]
    defaultInputModes:  List[str] = field(default_factory=lambda: ["text"])
    defaultOutputModes: List[str] = field(default_factory=lambda: ["text"])

# ── Task / Message ───────────────────────────────────────────────────
@dataclass
class Message:
    role: str
    content: str
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

@dataclass
class Task:
    id: str
    message: Message
    status: str = "submitted"  # submitted | working | completed | failed
    result: Optional[str] = None
    history: List[Message] = field(default_factory=list)

print("✅ A2A data types defined")

## Step 4: ワーカーエージェントの実装（3体）

各エージェントは以下のA2Aエンドポイントを持ちます:
- `GET  /.well-known/agent.json` — エージェントカードの公開
- `POST /tasks/send`            — タスクの受信・実行
- `POST /tasks/get`             — タスク状態の確認

In [ ]:
def make_worker_app(card: AgentCard, system_prompt: str) -> Flask:
    """ワーカーエージェント共通のFlaskアプリを生成する"""
    app = Flask(card.name)
    CORS(app)
    tasks: Dict[str, Task] = {}

    @app.route("/.well-known/agent.json")
    def agent_card_endpoint():
        """A2A仕様: エージェントカードを返す"""
        return jsonify(asdict(card))

    @app.route("/tasks/send", methods=["POST"])
    def tasks_send():
        """A2A仕様: タスクを受信して実行する"""
        body = request.get_json()
        task_id   = body.get("id", str(uuid.uuid4()))
        user_text = body["message"]["content"]

        task = Task(
            id=task_id,
            message=Message(role="user", content=user_text),
            status="working",
        )
        task.history.append(task.message)
        tasks[task_id] = task

        # ローカルLLMで推論
        response_text = llm_call(user_text, system_prompt=system_prompt)

        agent_msg = Message(role="agent", content=response_text)
        task.history.append(agent_msg)
        task.result = response_text
        task.status = "completed"

        return jsonify({
            "id":      task.id,
            "status":  task.status,
            "result":  asdict(agent_msg),
            "history": [asdict(m) for m in task.history],
        })

    @app.route("/tasks/get", methods=["POST"])
    def tasks_get():
        """A2A仕様: タスクの状態を返す"""
        body    = request.get_json()
        task_id = body.get("id")
        task    = tasks.get(task_id)
        if not task:
            return jsonify({"error": "Task not found"}), 404
        return jsonify({
            "id":     task.id,
            "status": task.status,
            "result": {"role": "agent", "content": task.result},
        })

    return app


# ── SummarizerAgent ──────────────────────────────────────────────────
summarizer_card = AgentCard(
    name="SummarizerAgent",
    description="Summarizes text into 3 concise bullet points.",
    url="http://localhost:8101",
    version="1.0.0",
    capabilities=AgentCapabilities(),
    skills=[AgentSkill(
        id="summarize",
        name="Text Summarization",
        description="Condenses any text into 3 bullet points"
    )]
)
summarizer_system = (
    "You are a summarization agent. "
    "Summarize the given text into exactly 3 concise bullet points. "
    "Output only the bullet points starting with '- ', nothing else."
)

# ── TranslatorAgent ───────────────────────────────────────────────────
translator_card = AgentCard(
    name="TranslatorAgent",
    description="Translates English text into Japanese.",
    url="http://localhost:8102",
    version="1.0.0",
    capabilities=AgentCapabilities(),
    skills=[AgentSkill(
        id="translate_ja",
        name="English to Japanese Translation",
        description="Translates English text into natural Japanese"
    )]
)
translator_system = (
    "You are a translation agent. "
    "Translate the given English text into natural Japanese. "
    "Output only the Japanese translation, nothing else."
)

# ── KeywordAgent ──────────────────────────────────────────────────────
keyword_card = AgentCard(
    name="KeywordAgent",
    description="Extracts the top 5 keywords from text.",
    url="http://localhost:8103",
    version="1.0.0",
    capabilities=AgentCapabilities(),
    skills=[AgentSkill(
        id="extract_keywords",
        name="Keyword Extraction",
        description="Extracts the 5 most important keywords from text"
    )]
)
keyword_system = (
    "You are a keyword extraction agent. "
    "Extract exactly 5 important keywords from the given text. "
    "Output only a comma-separated list of keywords, nothing else."
)


def run_flask(app, port):
    import logging
    log = logging.getLogger("werkzeug")
    log.setLevel(logging.ERROR)  # Flaskの冗長ログを抑制
    app.run(port=port, debug=False, use_reloader=False)

# ワーカー3体を別スレッドで起動
for card, system, port in [
    (summarizer_card, summarizer_system, 8101),
    (translator_card, translator_system, 8102),
    (keyword_card,    keyword_system,    8103),
]:
    app = make_worker_app(card, system)
    threading.Thread(target=run_flask, args=(app, port), daemon=True).start()

time.sleep(2)
print("✅ SummarizerAgent → http://localhost:8101")
print("✅ TranslatorAgent → http://localhost:8102")
print("✅ KeywordAgent    → http://localhost:8103")

## Step 5: エージェントカードの取得と表示

A2A仕様では `/.well-known/agent.json` を叩いてエージェントの能力を発見します。  
**HTTPエンドポイントへの実際のリクエスト**と**print表示**の両方を実施します。

In [ ]:
def fetch_agent_card(base_url: str) -> dict:
    """/.well-known/agent.json を叩いてエージェントカードを取得する"""
    url = f"{base_url}/.well-known/agent.json"
    resp = requests.get(url, timeout=5)
    resp.raise_for_status()
    return resp.json()

def print_agent_card(card: dict, show_raw: bool = False):
    """エージェントカードを整形表示する"""
    print(f"\n{'═'*55}")
    print(f"  🤖  {card['name']}  (v{card['version']})")
    print(f"  📍  {card['url']}")
    print(f"  📝  {card['description']}")
    print(f"  🔧  Skills:")
    for s in card['skills']:
        print(f"       [{s['id']}]  {s['name']}")
        print(f"               {s['description']}")
        print(f"               input={s['inputModes']}  output={s['outputModes']}")
    caps = card['capabilities']
    print(f"  ⚙️   streaming={caps['streaming']}  "
          f"pushNotifications={caps['pushNotifications']}  "
          f"stateHistory={caps['stateTransitionHistory']}")
    print(f"{'═'*55}")
    if show_raw:
        print("\n  📋  Raw JSON (A2A仕様準拠):")
        print(json.dumps(card, indent=4, ensure_ascii=False))


# ── 全ワーカーのカードをHTTPで取得して表示 ──
print("📡 /.well-known/agent.json を叩いてカードを取得中...\n")

worker_ports = [
    (8101, "SummarizerAgent"),
    (8102, "TranslatorAgent"),
    (8103, "KeywordAgent"),
]

for port, label in worker_ports:
    card = fetch_agent_card(f"http://localhost:{port}")
    print_agent_card(card)

# ── SummarizerのカードはRaw JSONも表示 ──
print("\n" + "─"*55)
print("📋 SummarizerAgent の Raw JSON（A2A仕様準拠）:")
raw = fetch_agent_card("http://localhost:8101")
print(json.dumps(raw, indent=4, ensure_ascii=False))

## Step 6: オーケストレーターエージェントの実装

オーケストレーターは:
1. 起動時にワーカーのエージェントカードを収集してレジストリに登録
2. ユーザーのタスクをLLMで解釈し、適切なワーカーへ委譲
3. 必要に応じてチェーン実行（前の出力を次の入力に）
4. 全結果をまとめて返す

In [ ]:
orchestrator_app = Flask("OrchestratorAgent")
CORS(orchestrator_app)

WORKER_URLS = [
    "http://localhost:8101",
    "http://localhost:8102",
    "http://localhost:8103",
]

# ── エージェントレジストリ ────────────────────────────────────────────
worker_registry: Dict[str, dict] = {}  # name → {card, url}

def discover_workers():
    """起動時にワーカーのエージェントカードを収集してレジストリへ登録"""
    print("\n🔍 エージェントカードを収集中...")
    for url in WORKER_URLS:
        try:
            card = fetch_agent_card(url)
            worker_registry[card["name"]] = {"card": card, "url": url}
            skill_ids = [s["id"] for s in card["skills"]]
            print(f"  ✅ {card['name']} @ {url}  skills={skill_ids}")
        except Exception as e:
            print(f"  ❌ {url}: {e}")
    print(f"  → レジストリ登録数: {len(worker_registry)} エージェント")


def delegate_task(worker_name: str, content: str) -> dict:
    """ワーカーへA2Aタスクを委譲する (tasks/send)"""
    worker = worker_registry.get(worker_name)
    if not worker:
        return {"error": f"{worker_name} not found"}
    payload = {
        "id":      str(uuid.uuid4()),
        "message": {"role": "user", "content": content}
    }
    r = requests.post(
        f"{worker['url']}/tasks/send",
        json=payload, timeout=120
    )
    return r.json()


# ── ルーティングロジック ───────────────────────────────────────────────
ROUTING_SYSTEM = """You are a task routing agent.
Available agents:
- SummarizerAgent: summarizes text into bullet points
- TranslatorAgent: translates English text to Japanese
- KeywordAgent: extracts keywords from text

Based on the user's request, decide which agents to use and in what order.
Respond with ONLY a JSON array of agent names in execution order.
Example: ["SummarizerAgent", "TranslatorAgent"]"""

ALL_AGENTS = ["SummarizerAgent", "TranslatorAgent", "KeywordAgent"]

def route_task(user_text: str) -> List[str]:
    """LLMでルーティングを判定し、実行するエージェント名リストを返す"""
    raw = llm_call(user_text, system_prompt=ROUTING_SYSTEM, max_new_tokens=64)
    # LLMの出力からエージェント名を抽出（堅牢なパース）
    agents = [a for a in ALL_AGENTS if a in raw]
    # 順序をraw内の出現位置で並べ替え
    agents.sort(key=lambda a: raw.find(a))
    return agents if agents else ["SummarizerAgent"]  # デフォルトフォールバック


# ── オーケストレーター自身のエージェントカード ──
orch_card = AgentCard(
    name="OrchestratorAgent",
    description="Routes and orchestrates tasks across Summarizer, Translator, and Keyword agents.",
    url="http://localhost:8100",
    version="1.0.0",
    capabilities=AgentCapabilities(stateTransitionHistory=True),
    skills=[AgentSkill(
        id="orchestrate",
        name="Task Orchestration",
        description="Dynamically routes and chains tasks across worker agents"
    )]
)

@orchestrator_app.route("/.well-known/agent.json")
def orch_card_endpoint():
    return jsonify(asdict(orch_card))

@orchestrator_app.route("/tasks/send", methods=["POST"])
def orch_tasks_send():
    body      = request.get_json()
    task_id   = body.get("id", str(uuid.uuid4()))
    user_text = body["message"]["content"]

    log = []

    # 1. ルーティング判定
    agents_to_call = route_task(user_text)
    log.append({"step": "routing", "agents_selected": agents_to_call})

    # 2. チェーン委譲（前の出力を次の入力に渡す）
    current_text = user_text
    intermediate: Dict[str, str] = {}

    for agent_name in agents_to_call:
        resp        = delegate_task(agent_name, current_text)
        result_text = resp.get("result", {}).get("content", "")
        intermediate[agent_name] = result_text
        current_text = result_text  # チェーン: 前の出力 → 次の入力
        log.append({"step": f"delegated_to_{agent_name}", "output": result_text})

    return jsonify({
        "id":                   task_id,
        "status":               "completed",
        "result":               {"role": "agent", "content": current_text},
        "log":                  log,
        "intermediate_results": intermediate,
    })


threading.Thread(
    target=run_flask, args=(orchestrator_app, 8100), daemon=True
).start()
time.sleep(2)

discover_workers()
print("\n✅ OrchestratorAgent → http://localhost:8100")

## Step 7: 全エージェントのカードを一覧表示（オーケストレーター含む）

In [ ]:
print("📡 全エージェントのカード一覧\n")
for port, name in [(8100, "Orchestrator"), (8101, "Summarizer"),
                   (8102, "Translator"),   (8103, "Keyword")]:
    card = fetch_agent_card(f"http://localhost:{port}")
    print_agent_card(card)

## Step 8: A2Aデモ実行

ユーザーのリクエストをオーケストレーターに送り、A2A委譲の流れを観察します。

In [ ]:
def run_demo(user_input: str, label: str = ""):
    """A2Aデモを実行して委譲フローと結果を表示する"""
    print(f"\n{'━'*60}")
    print(f"  🧑 ユーザー入力  [{label}]")
    print(f"  {user_input}")
    print(f"{'━'*60}")

    resp = requests.post(
        "http://localhost:8100/tasks/send",
        json={"id": str(uuid.uuid4()),
              "message": {"role": "user", "content": user_input}},
        timeout=300,
    )
    data = resp.json()

    print("\n  📋 A2A委譲フロー:")
    for step in data.get("log", []):
        s = step["step"]
        if s == "routing":
            print(f"    🔀 ルーティング → {step['agents_selected']}")
        elif s.startswith("delegated_to_"):
            agent = s.replace("delegated_to_", "")
            out   = step['output'][:120].replace("\n", " ")
            print(f"    ✉️  {agent}")
            print(f"       └ {out}{'...' if len(step['output']) > 120 else ''}")

    print(f"\n  🤖 最終出力:")
    for line in data["result"]["content"].split("\n"):
        print(f"     {line}")

    return data


# ── デモ1: 要約のみ ────────────────────────────────────────────────
d1 = run_demo(
    "Please summarize the following text: "
    "The A2A protocol is an open standard by Google that enables AI agents to "
    "communicate, delegate tasks, and collaborate across different platforms. "
    "It defines how agents discover each other via Agent Cards and exchange "
    "structured messages using tasks/send and tasks/get endpoints.",
    label="要約"
)

In [ ]:
# ── デモ2: キーワード抽出 + 翻訳チェーン ──────────────────────────
d2 = run_demo(
    "Extract keywords and translate them to Japanese: "
    "Machine learning is a subset of artificial intelligence that enables "
    "systems to learn from data, identify patterns, and make decisions "
    "with minimal human intervention.",
    label="キーワード抽出→翻訳チェーン"
)

In [ ]:
# ── デモ3: 要約→翻訳チェーン ──────────────────────────────────────
d3 = run_demo(
    "Summarize and translate to Japanese: "
    "Context window efficiency has become a critical concern for AI agents. "
    "Tools like rtk compress CLI output before passing it to the LLM, "
    "reducing token consumption by up to 89 percent and extending session "
    "duration by approximately three times.",
    label="要約→翻訳チェーン"
)

## Step 9: ワーカーへの直接呼び出し + tasks/get による状態確認

In [ ]:
# ── tasks/send で直接送信 ──────────────────────────────────────────
print("🔧 KeywordAgent への直接A2A呼び出し")
print("─"*55)

direct_id  = str(uuid.uuid4())
direct_req = {
    "id":      direct_id,
    "message": {"role": "user",
                "content": "Agent cards, A2A protocol, task delegation, "
                           "orchestration, context window efficiency."}
}
r    = requests.post("http://localhost:8103/tasks/send", json=direct_req, timeout=120)
resp = r.json()

print(f"  Task ID : {resp['id']}")
print(f"  Status  : {resp['status']}")
print(f"  Result  : {resp['result']['content']}")

print("\n📜 stateTransitionHistory（タスク履歴）:")
for msg in resp.get("history", []):
    icon = "🧑" if msg["role"] == "user" else "🤖"
    print(f"  {icon} [{msg['role']}]  {msg['content'][:80]}")
    print(f"      timestamp: {msg['timestamp']}")

# ── tasks/get で状態を後から確認 ──────────────────────────────────
print("\n🔍 tasks/get で状態確認:")
r2   = requests.post("http://localhost:8103/tasks/get", json={"id": direct_id})
data = r2.json()
print(f"  id     : {data['id']}")
print(f"  status : {data['status']}")
print(f"  result : {data['result']['content']}")

## まとめ

このデモで確認したA2A仕様の要素:

| A2A要素 | 実装 |
|---|---|
| **AgentCard** | `/.well-known/agent.json` をHTTPで公開 |
| **エージェント発見** | オーケストレーターが起動時にカードを収集・レジストリ化 |
| **tasks/send** | タスクの委譲エンドポイント |
| **tasks/get** | タスク状態の非同期確認 |
| **stateTransitionHistory** | メッセージ履歴（user/agent/timestamp）の記録 |
| **チェーン委譲** | 前エージェントの出力を次の入力に渡す連鎖実行 |
| **ローカルLLM** | Transformers (`Qwen2.5-0.5B-Instruct`) による推論 |

### 発展的な試み
- モデルを `Qwen2.5-1.5B-Instruct` や `google/gemma-3-1b-it` に変更してルーティング精度を比較する
- 新しいワーカー（例: SentimentAgent）を追加して `discover_workers()` が自動認識することを確認する
- `streaming: true` のエージェントカードでストリーミング応答を実装する